In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [2]:
# 1. Setup
SEED = 42
np.random.seed(SEED)

N_TOPICS = 5

print("LDA Setup Complete")

LDA Setup Complete


In [3]:
# 2. Load Dataset
dataset = load_dataset("SetFit/bbc-news")

train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()

X = train["text"].fillna("").to_numpy()

X_train, X_val = train_test_split(
    X,
    test_size=0.2,
    random_state=SEED
)

X_test = test["text"].fillna("").to_numpy()

print("Dataset Loaded")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))



Dataset Loaded
Training: 980
Validation: 245
Testing: 1000


In [4]:
# 3. Text Representation using Bag of Words
lda_vectorizer = CountVectorizer(
    max_features=10000,
    stop_words="english",
    max_df=0.95,
    min_df=2
)

X_train_bow = lda_vectorizer.fit_transform(X_train)
X_val_bow = lda_vectorizer.transform(X_val)
X_test_bow = lda_vectorizer.transform(X_test)

print("Bag of Words Complete")
print("Document-term matrix:", X_train_bow.shape)

Bag of Words Complete
Document-term matrix: (980, 10000)


In [5]:
# 4. Train LDA Model
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=SEED,
    max_iter=10,
    learning_method="batch"
)

lda_model.fit(X_train_bow)

print("LDA Training Complete")

LDA Training Complete


In [6]:
# 5. Display Top Words for Each Topic
feature_names = lda_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda_model.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices]

    print(f"\nTopic {topic_idx + 1}:")
    print(", ".join(top_words))


Topic 1:
said, mr, government, labour, party, election, blair, minister, people, new

Topic 2:
said, year, film, world, won, number, best, new, time, win

Topic 3:
said, year, growth, england, economy, market, bank, 2004, half, sales

Topic 4:
said, music, people, new, tv, best, game, like, uk, games

Topic 5:
said, people, new, mobile, company, firm, users, mr, net, service


In [7]:
# 6. Topic Distribution for Each Document
lda_train_features = lda_model.transform(X_train_bow)
lda_val_features = lda_model.transform(X_val_bow)
lda_test_features = lda_model.transform(X_test_bow)

print("\nTrain topic features:", lda_train_features.shape)
print("Validation topic features:", lda_val_features.shape)
print("Test topic features:", lda_test_features.shape)

print("\nTopic distribution for first training article:")
print(np.round(lda_train_features[0], 3))



Train topic features: (980, 5)
Validation topic features: (245, 5)
Test topic features: (1000, 5)

Topic distribution for first training article:
[0.05  0.001 0.311 0.019 0.619]


In [8]:
# 7. Evaluate Perplexity
train_perplexity = lda_model.perplexity(X_train_bow)
test_perplexity = lda_model.perplexity(X_test_bow)

print("\nLDA Evaluation")
print(f"Train Perplexity: {train_perplexity:.4f}")
print(f"Test Perplexity : {test_perplexity:.4f}")



LDA Evaluation
Train Perplexity: 3417.2217
Test Perplexity : 4043.0827


In [ ]:
# 8. Evaluate Topic Coherence
# Tokenisasi sederhana untuk menghitung coherence
tokenized_texts = [
    lda_vectorizer.build_analyzer()(text)
    for text in X_train
]

dictionary = Dictionary(tokenized_texts)

topic_words = []

for topic in lda_model.components_:
    top_indices = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_indices]
    topic_words.append(words)

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()

print(f"Topic Coherence (C_v): {coherence_score:.4f}")

In [ ]:
# 9. Visualize Topic Distribution
topic_distribution = lda_train_features.mean(axis=0)

plt.figure(figsize=(8, 5))
plt.bar(
    range(1, N_TOPICS + 1),
    topic_distribution
)

plt.title("Average Topic Distribution")
plt.xlabel("Topic")
plt.ylabel("Average Probability")
plt.xticks(range(1, N_TOPICS + 1))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 10. Create Topic Distribution Table
topic_columns = [
    f"Topic_{i + 1}"
    for i in range(N_TOPICS)
]

topic_df = pd.DataFrame(
    lda_train_features,
    columns=topic_columns
)

topic_df["Dominant_Topic"] = (
    lda_train_features.argmax(axis=1) + 1
)

print("\nTopic Distribution Table:")
print(topic_df.head())

In [ ]:
# 11. Display Example Document and Its Topics
sample_idx = 0

print("\nExample Document:")
print(X_train[sample_idx][:500])

print("\nTopic Probabilities:")

for i, probability in enumerate(lda_train_features[sample_idx]):
    print(f"Topic {i + 1}: {probability:.2%}")

print(
    "\nDominant Topic:",
    lda_train_features[sample_idx].argmax() + 1
)

print("\nLDA Experiment Complete")